# Tutorial 5: Train NicheTrans on STARmap PLUS data

In [8]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_img import *
from datasets.data_manager_STARmap_PLUS import AD_Mouse

from utils.utils import *
from utils.utils_training_STARmap_PLUS import train, test
from utils.utils_dataloader import *
from prior_AddOn.gene_embedding_loader import load_static_gene_prior
from prior_AddOn.gene_prior_filter import filter_dataset_by_gene_prior

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [9]:
%run ./args/args_STARmap_PLUS.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.5, dropout_rate=0.25, n_top_genes=2000, workers=4, AD_adata_path='/home1/shezixi/data/in_silico_central_dogma_challenge/unprocessed/2023_nn_AD_mouse/AD_model_adata_protein', Wild_type_adata_path='/home1/shezixi/data/in_silico_central_dogma_challenge/unprocessed/2023_nn_AD_mouse/wild_type_adata_protein', max_epoch=40, stepsize=20, train_batch=128, test_batch=32, optimizer='adam', lr=0.0001, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [10]:
# create the dataset
dataset = AD_Mouse(AD_adata_path=args.AD_adata_path, Wild_type_adata_path=args.Wild_type_adata_path, n_top_genes=args.n_top_genes)

# load and apply static gene priors before creating dataloaders/model
prior_model = "scgpt"
priors = load_static_gene_prior(
    source_panel=dataset.source_panel,
    models=(prior_model,),
    species="mouse",
    root="prior_AddOn/gene_embeddings",
)
dataset, priors, filter_info = filter_dataset_by_gene_prior(
    dataset=dataset,
    priors=priors,
    prior_model=prior_model,
)
print(
    f"Gene prior filtering ({prior_model}): kept "
    f"{len(filter_info['filtered_source_panel'])}/"
    f"{len(filter_info['original_source_panel'])} genes; "
    f"removed {len(filter_info['removed_genes'])}."
)

# create the dataloaders after filtering so feature dimensions stay aligned
trainloader, testloader, _ = ad_mouse_dataloader(args, dataset)

# create the model
source_dimension, target_dimension = dataset.rna_length, dataset.target_length
model = NicheTrans(
    source_length=source_dimension,
    target_length=target_dimension,
    noise_rate=args.noise_rate,
    dropout_rate=args.dropout_rate,
    priors=priors,
    prior_model=prior_model,
    prior_pooling_mode="qkv",
)
model = nn.DataParallel(model).cuda()


------Calculating spatial graph...
The graph contains 124464 edges, 10372 cells.
12.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 115608 edges, 9634 cells.
12.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 96408 edges, 8034 cells.
12.0000 neighbors per cell on average.
=> AD Mouse loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |  10372 spots, 894.0 positive tao, 291.0 positive plaque 
  test     |   9634 spots, 620.0 positive tao, 195.0 positive plaque 
  ------------------------------
Gene prior filtering (scgpt): kept 1596/1719 genes; removed 123.


### Load bio-prior

In [11]:
# inspect filtering details if needed
filter_info["coverage_before"], filter_info["coverage_after"]


({'model': 'scgpt',
  'species': 'mouse',
  'n_features': 1719,
  'n_found': 1596,
  'coverage': 0.9284467713787086,
  'by_status': {'mapped': 1596, 'unmapped': 120, 'missing_embedding': 3}},
 {'model': 'scgpt',
  'species': 'mouse',
  'n_features': 1596,
  'n_found': 1596,
  'coverage': 1.0,
  'by_status': {'mapped': 1596}})

### Initialize loss function (criterion) and optimizer

In [12]:
criterion = nn.BCELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [13]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader)
    if args.stepsize > 0: scheduler.step()
    ################

pearson = test(model, testloader)
torch.save(model.state_dict(), 'NicheTrans_STARmap_PLUS.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/40
Batch 81/81	 Loss 0.526278 (0.612868)
==> Epoch 2/40
Batch 81/81	 Loss 0.411802 (0.460868)
==> Epoch 3/40
Batch 81/81	 Loss 0.333213 (0.362782)
==> Epoch 4/40
Batch 81/81	 Loss 0.265792 (0.296000)
==> Epoch 5/40
Batch 81/81	 Loss 0.224837 (0.248530)
==> Epoch 6/40
Batch 81/81	 Loss 0.225972 (0.215533)
==> Epoch 7/40
Batch 81/81	 Loss 0.175125 (0.189449)
==> Epoch 8/40
Batch 81/81	 Loss 0.163461 (0.170643)
==> Epoch 9/40
Batch 81/81	 Loss 0.139292 (0.155674)
==> Epoch 10/40
Batch 81/81	 Loss 0.158613 (0.149051)
==> Epoch 11/40
Batch 81/81	 Loss 0.130412 (0.140689)
==> Epoch 12/40
Batch 81/81	 Loss 0.126231 (0.134149)
==> Epoch 13/40
Batch 81/81	 Loss 0.112432 (0.126630)
==> Epoch 14/40
Batch 81/81	 Loss 0.098771 (0.121931)
==> Epoch 15/40
Batch 81/81	 Loss 0.088632 (0.120076)
==> Epoch 16/40
Batch 81/81	 Loss 0.077268 (0.115174)
==> Epoch 17/40
Batch 81/81	 Loss 0.113604 (0.109391)
==> Epoch 18/40
Batch 81/81	 Loss 0.116786 (0.108260)
==> Epoch 19/40
Batch 81/81	 Loss 0.0

In [14]:
for row in priors["scgpt"]["mapping_table"]:
    if row["status"] != "mapped":
        print(row["input_gene"], row["status"], row["reason"])